<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Censo Agropecuario: Revisar el Cuaderno Finca por Finca 📋 (Para Dummies)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/07%20-%20Temas%20Avanzados%20en%20Data%20Analysis/Para%20Dummies/04_Aplicacion_EDA_Censo_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 04 del Módulo 07 (Aplicación EDA: Censo Nacional Agropecuario)](../04_Aplicacion_EDA_Censo_Nacional_Agropecuario.ipynb). Un **censo agropecuario** es como que un funcionario pase **finca por finca** de todo el país preguntando: ¿cuánta tierra tiene?, ¿qué siembra?, ¿tiene crédito? Después los **cuadernos** de miles de funcionarios se juntan en una sola tabla gigante. Y como todo cuaderno escrito a mano, **trae errores**: un cero de más, un «999» donde no sabían la edad, el nombre de la vereda escrito de tres maneras.

| En el censo de la vereda | En el computador |
|---|---|
| Revisar el cuaderno antes de sacar cuentas | **Perfilar y limpiar** los datos |
| «Esta finca dice tener 25 hectáreas sembradas y solo mide 2,5» | Una **regla de sentido común** que marca el error |
| «Quién tiene la tierra» | La **concentración** (cuánta tierra tiene el 10 % más grande) |
| Datos **inventados** para practicar | Un archivo **sintético**; **no** es del DANE |

> ⚠️ **Todos los datos de este cuaderno son inventados.** No son los del censo real; sirven para aprender el método. En el cuaderno estándar hay una guía para **bajar y adaptar el censo real**.

Al terminar podrás explicar:
1. Por qué **hay que revisar el cuaderno antes de sacar el promedio** (y cuánto lo mueven unos pocos errores).
2. Cómo **«marcar»** los datos imposibles en lugar de borrarlos a escondidas.
3. Cómo se ve la **desigualdad en el reparto de la tierra** y por qué el promedio de hectáreas no cuenta toda la historia.

> 🔗 Si luego quieres la versión **con reglas, bitácora, Gini, intervalos y datos faltantes**: [cuaderno estándar](../04_Aplicacion_EDA_Censo_Nacional_Agropecuario.ipynb). Taller de todo el módulo: [`../../homeworks/Para Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb).

---
## 1. Revisar el Cuaderno Antes de Sacar Cuentas 🧹

Inventamos el cuaderno de una vereda con **300 fincas** y le dejamos **errores típicos**: seis fincas con **un cero de más** en las hectáreas sembradas, cinco edades escritas como **999**, y el nombre de la vereda escrito en mayúsculas o con espacios de sobra. Veamos cuánto engañan y cómo se arreglan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(8)
n = 300

# Cuaderno de la vereda con 300 fincas INVENTADAS
fincas = pd.DataFrame({"vereda": rng.choice(["La Esperanza", "El Retiro", "Alto Bonito"], n),
                       "hectareas": rng.lognormal(0.7, 1.0, n).round(2)})
fincas["sembrada"] = (fincas["hectareas"] * rng.uniform(0.2, 0.8, n)).round(2)
fincas["edad"] = rng.normal(52, 12, n).round(0)
verdad = fincas.copy()                                                    # lo que de verdad pasó

# El cuaderno llega con errores de digitación (lo normal en la vida real)
fincas.loc[rng.choice(n, 6, replace=False), "sembrada"] *= 10             # un cero de más
fincas.loc[rng.choice(n, 5, replace=False), "edad"] = 999                 # "no sabe" escrito como 999
i = rng.choice(n, 60, replace=False)
fincas.loc[i[:30], "vereda"] = fincas.loc[i[:30], "vereda"].str.upper()   # escrito en mayúsculas
fincas.loc[i[30:], "vereda"] = " " + fincas.loc[i[30:], "vereda"] + " "   # con espacios de más

print("Veredas distintas según el cuaderno:", fincas["vereda"].nunique(), "(pero en realidad son 3)")
fincas["vereda"] = fincas["vereda"].str.strip().str.lower()
print("Después de ordenar la escritura:", fincas["vereda"].nunique())

# Reglas de sentido común: se MARCAN, no se borran a escondidas
mas_sembrada_que_finca = fincas["sembrada"] > fincas["hectareas"]
edad_imposible = ~fincas["edad"].between(15, 100)
print("Fincas con más sembrado que hectáreas:", mas_sembrada_que_finca.sum(), "| edades imposibles:", edad_imposible.sum())

print(f"Promedio de hectáreas sembradas: verdad = {verdad['sembrada'].mean():.2f} | con errores = {fincas['sembrada'].mean():.2f}")
print(f"Edad promedio: verdad = {verdad['edad'].mean():.1f} | con errores = {fincas['edad'].mean():.1f}")


---
**Lección:** el computador creyó que había **9 veredas** cuando solo hay **3**: «El Retiro», «EL RETIRO» y « El Retiro » son para él nombres distintos. Basta quitar espacios y pasar todo a minúsculas para que vuelvan a ser **3**. Las reglas de sentido común **encontraron exactamente los errores que metimos**: **6** fincas con más sembrado que hectáreas y **5** edades imposibles. Y fíjate cuánto mueven el promedio unos pocos errores: la edad promedio pasa de **51.1** años (la verdad) a **66.9** (¡por solo 5 fincas de 300!), y el sembrado promedio de **1.75** a **2.15** hectáreas. Por eso **se revisa antes de promediar**, y los datos imposibles se **marcan** (para revisarlos o corregirlos con el funcionario) en vez de borrarse a escondidas.

---
## 2. ¿Quién Tiene la Tierra? 🥧

Con las cifras de la verdad, ordenamos las 300 fincas **de la más pequeña a la más grande** y las partimos en **10 grupos de 30 fincas**. Si la tierra estuviera repartida parejo, cada grupo tendría el **10 %**. ¿Qué pasa en la vereda inventada?

In [ ]:
# Repartir "la torta de la tierra": ordenamos las fincas de la más pequeña a la más grande
ha = np.sort(verdad["hectareas"].to_numpy())
total = ha.sum()
decil = np.array([g.sum() for g in np.array_split(ha, 10)]) / total * 100          # % de la tierra que tiene cada grupo de 30 fincas
print(f"El 10 % de las fincas más grandes tiene el {decil[-1]:.0f} % de la tierra; la mitad más pequeña, solo el {decil[:5].sum():.0f} %")
print(f"Finca típica (mediana): {np.median(ha):.2f} ha | finca promedio: {ha.mean():.2f} ha")

plt.figure(figsize=(7.5, 4))
plt.bar(range(1, 11), decil, color=["#93c5fd"] * 9 + ["#1e3a8a"])
plt.axhline(10, color="grey", ls="--", label="reparto parejo (10 % cada grupo)")
plt.xlabel("grupos de fincas, de las más pequeñas (1) a las más grandes (10)"); plt.ylabel("% de la tierra"); plt.legend(); plt.title("¿Quién tiene la tierra? (datos inventados)"); plt.show()


---
**Lección:** el **10 % de las fincas más grandes** (30 fincas) tiene el **40 %** de la tierra, mientras que la **mitad más pequeña** (150 fincas) tiene solo el **15 %**. Eso es **concentración**: unos pocos tienen mucho y muchos tienen poco. Fíjate también en que la finca **típica** (la del medio) mide **1.98 ha**, pero la finca **promedio** mide **3.49 ha**: las fincas grandes **jalan el promedio hacia arriba**. Por eso, cuando hay pocos muy grandes, conviene mirar **el valor del medio y el reparto**, no solo el promedio.

---
##### 🛠️ Práctica 1: El 5 % más grande

Con `ha` (las hectáreas de las 300 fincas, ordenadas de menor a mayor) y `total` (la suma de todas):
1. Calcula **cuántas fincas** son el 5 % del total (`cinco`).
2. Calcula qué **porcentaje de la tierra** tienen esas fincas más grandes (`pct_5`).
3. Comprueba con `assert` que es **más del triple** de lo que tendrían con un reparto parejo (5 %).

In [ ]:
# Escribe tu código aquí

# cinco = ...
# pct_5 = ha[-cinco:].sum() / total * 100

# assert pct_5 > 5 * 3


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
cinco = int(round(0.05 * len(ha)))                       # número de fincas en el 5 % más grande
pct_5 = ha[-cinco:].sum() / total * 100
print(f"El 5 % de las fincas más grandes ({cinco} fincas) tiene el {pct_5:.0f} % de la tierra")
assert pct_5 > 5 * 3                                     # mucho más que el reparto parejo (5 %)
```
</details>

---

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué el computador contó **9 veredas** cuando en realidad son 3? ¿Qué pasaría si sacaras «el promedio por vereda» sin arreglarlo?
2. Cinco edades «999» movieron la edad promedio de 51 a 67 años. ¿Qué harías con esas cinco fincas: borrarlas, corregirlas o marcarlas? ¿Por qué?
3. Si en tu municipio el 10 % de las fincas tiene el 40 % de la tierra, ¿qué te dice eso que no te diría el «promedio de hectáreas por finca»?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>